In [28]:
import pandas as pd
from pathlib import Path
import sys
sys.path.append("../src")
from preprocessing import create_preprocessor

FE_DF_PATH = Path("../data/processed/telco_customer_churn_feature_engineered.parquet")
fe_df = pd.read_parquet(FE_DF_PATH)

In [29]:
# look at the df
fe_df.head()

,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,NumServices,LongTimeCustomer,IsHighSpender,IsNewCustomer,TenureGroup,AutoPayment,HasSupportServices,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.850000,29.85,2,0,0,1,0-12,0,0,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.950001,1889.50,4,0,0,0,24-48,0,1,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.849998,108.15,4,0,0,1,0-12,0,1,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.299999,1840.75,4,0,0,0,24-48,1,1,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.699997,151.65,2,0,0,1,0-12,0,0,Yes


In [30]:
# info about the dataframe
fe_df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   Gender              7043 non-null   category
 1   SeniorCitizen       7043 non-null   int8    
 2   Partner             7043 non-null   category
 3   Dependents          7043 non-null   category
 4   Tenure              7043 non-null   int8    
 5   PhoneService        7043 non-null   category
 6   MultipleLines       7043 non-null   category
 7   InternetService     7043 non-null   category
 8   OnlineSecurity      7043 non-null   category
 9   OnlineBackup        7043 non-null   category
 10  DeviceProtection    7043 non-null   category
 11  TechSupport         7043 non-null   category
 12  StreamingTV         7043 non-null   category
 13  StreamingMovies     7043 non-null   category
 14  Contract            7043 non-null   category
 15  PaperlessBilling    7043 non-null   category
 16 

In [31]:
# create train and test set
from sklearn.model_selection import train_test_split

X = fe_df.drop("Churn", axis=1)
y = fe_df.Churn.map({"Yes": 1, "No": 0})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [33]:
# create numerical and categorical features
num_features = X.select_dtypes(include="number").columns
cat_features = X.select_dtypes(exclude="number").columns

In [34]:
# create preprocessor and pipelines
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

preprocessor = create_preprocessor(num_features, cat_features)

# handle class imbalance for XG Boost 
churn_count = y_train.sum()
normal_count = len(y_train) - churn_count
scale_pos_weight = normal_count / churn_count

pipelines = {
    "Dummy": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", DummyClassifier(strategy="most_frequent")),
    ]),
    "Logistic Regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced")),
    ]),
    "Random Forest": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced", n_jobs=-1)),
    ]),
    "XG Boost": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(n_estimators=100, random_state=42, scale_pos_weight=scale_pos_weight, eval_metric="logloss")),
    ]),
}

In [35]:
# metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score
from sklearn.model_selection import cross_val_score, StratifiedKFold
import joblib

results = {}

for model_name, pipeline in pipelines.items():
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(pipeline,
                                X_train,
                                y_train,
                                cv=cv,
                                scoring="f1")
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    results[model_name] = {
        "CV Mean F1": cv_scores.mean(),
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred),
        "ROC AUC Score": roc_auc_score(y_test, y_proba),
        "F1 Score": f1_score(y_test, y_pred) 
    }
    filename = model_name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(pipeline, f"../outputs/models/{filename}")

In [36]:
# save the metrics dataframe
results_df = pd.DataFrame(results).T
results_df.to_csv("../data/processed/metrics_result.csv", index=True)